In [3]:
from scipy.cluster.hierarchy import linkage, cut_tree, dendrogram
from collections import defaultdict

import json
from typing import List, Dict

import os
import sys

from sklearn.cluster import KMeans
import numpy as np

In [4]:
def carregar_embeddings_do_json(nome_arquivo) -> List[List[float]]:
    """
    Carrega embeddings de um arquivo JSON no formato especificado.

    Args:
        nome_arquivo (str): O caminho para o arquivo JSON.

    Returns:
        list or None: Uma lista de listas de floats (os embeddings),
                     ou None se ocorrer um erro ao carregar o arquivo.
    """
    try:
        with open(nome_arquivo, 'r') as f:
            data = json.load(f)
            if "embedding" in data and isinstance(data["embedding"], list):
                return data["embedding"]
            else:
                print(f"Formato inválido no arquivo '{nome_arquivo}'. Esperava uma lista na chave 'embedding'.")
                return None
    except FileNotFoundError:
        print(f"Arquivo '{nome_arquivo}' não encontrado.")
        return None
    except json.JSONDecodeError:
        print(f"Erro ao decodificar JSON do arquivo '{nome_arquivo}'.")
        return None
    except Exception as e:
        print(f"Ocorreu um erro ao carregar o arquivo '{nome_arquivo}': {e}")
        return None

In [5]:
nome_do_arquivo_json = r"app_duckdb\comments_1221_mistral.json"  # Substitua pelo nome do seu arquivo

embeddings_carregados = carregar_embeddings_do_json(nome_do_arquivo_json)

In [6]:
len(embeddings_carregados)

1221

In [7]:
def carregar_comentarios_de_txt(filename: str = "comentarios.txt") -> List[str]:
    """Carrega os comentários de um arquivo de texto e os retorna em uma lista."""
    comentarios_carregados = []
    try:
        with open(filename, 'r', encoding='utf-8') as f:
            for linha in f:
                comentarios_carregados.append(linha.strip())
        print(f"Comentários carregados com sucesso do arquivo '{filename}'")
    except FileNotFoundError:
        print(f"Arquivo '{filename}' não encontrado.")
    except IOError as e:
        print(f"Erro ao ler o arquivo '{filename}': {e}")
    return comentarios_carregados

In [8]:
all_comments = carregar_comentarios_de_txt()
all_comments[:5]

Comentários carregados com sucesso do arquivo 'comentarios.txt'


['3:03 "You can clap about that all you want. Enjoy". Get \'em Sam! Haha 😎😂',
 "I respect everyone's psition on the matter, in my opinion great interview by Sam.",
 'That last world Sam describes... If those kids get power outage for 1 week, they will die.',
 'Sam Altman never really answer. Maybe I am not enought inteligent to understand what is talking about but.....I feel he never responde',
 'AI is not about ego and who gets credit for what. Its about helping humanity evolve, push boundaries, and set higher standards for future generations. Its making non ego driven creative people a million times more creative']

In [9]:
def compute_clusters(n_clusters=3):
    complete_clustering = linkage(embeddings_carregados, 
        method="complete", metric="cosine")
    cluster_labels = cut_tree(complete_clustering, 
        n_clusters=n_clusters).reshape(-1, )

    groups = defaultdict(list)
    for id, label in zip(all_comments, cluster_labels):
        groups[label].append(id)
    return groups, cluster_labels

In [10]:
import numpy as np
from tqdm import tqdm

# 1. Verifica e filtra embeddings inconsistentes
valid_embeddings = []
valid_comments = []
invalid_indices = []

for idx, (comment, emb) in tqdm(enumerate(zip(all_comments, embeddings_carregados)), total=len(all_comments)):
    if isinstance(emb, (list, np.ndarray)) and len(emb) > 0:  # Verifica se é um embedding válido
        valid_embeddings.append(emb)
        valid_comments.append(comment)
    else:
        invalid_indices.append(idx)

print(f"\nEmbeddings válidos: {len(valid_embeddings)}")
print(f"Embeddings inválidos: {len(invalid_indices)}")
print(f"Exemplo de índices inválidos: {invalid_indices[:5]}")

# 2. Padroniza as dimensões (opcional - preenche com zeros se necessário)
max_dim = max(len(emb) for emb in valid_embeddings)
embeddings_padded = []
for emb in valid_embeddings:
    if len(emb) < max_dim:
        # Preenche com zeros se for menor que a dimensão máxima
        padded = np.pad(emb, (0, max_dim - len(emb)), 'constant')
        embeddings_padded.append(padded)
    else:
        embeddings_padded.append(emb)

# 3. Converte para numpy array
embeddings_array = np.array(embeddings_padded)
print(f"\nShape final do array: {embeddings_array.shape}")

# Filtra apenas os embeddings válidos (não vazios e com mesma dimensão)
reference_dim = len(embeddings_array[0])  # Assume primeiro como referência
filtered_data = [
    (comment, emb) 
    for comment, emb in zip(all_comments, embeddings_array) 
    if isinstance(emb, (list, np.ndarray)) and len(emb) == reference_dim
]

filtered_comments, filtered_embeddings = zip(*filtered_data)
embeddings_array = np.array(filtered_embeddings)

100%|██████████| 1221/1221 [00:00<00:00, 405161.80it/s]


Embeddings válidos: 1211
Embeddings inválidos: 10
Exemplo de índices inválidos: [171, 173, 175, 177, 179]

Shape final do array: (1211, 1024)


### Clusterização Básica K-Means

In [11]:
# Converte para numpy array se já não for
embeddings_array = np.array(embeddings_array)

# Clusterização com K-Means (5 clusters por padrão)
n_clusters = 5
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
cluster_labels = kmeans.fit_predict(embeddings_array)

# Visualização rápida
for i in range(n_clusters):
    print(f"\nCluster {i} - {sum(cluster_labels == i)} comentários:")
    print("Exemplos:", [all_comments[j][:50] + "..." for j in np.where(cluster_labels == i)[0][:3]])


Cluster 0 - 1 comentários:
Exemplos: ['I haven’t seen a Ted Talk I didn’t like. But this ...']

Cluster 1 - 349 comentários:
Exemplos: ['AI is not about ego and who gets credit for what. ...', 'If not already possible, I believe within the next...', 'My honeymoon period with ChatGPT came to an end re...']

Cluster 2 - 261 comentários:
Exemplos: ['Sam Altman never really answer. Maybe I am not eno...', "Although I don't believe he's necessarily an evil ...", 'sam is cracking a bit, couple blunders in this one...']

Cluster 3 - 326 comentários:
Exemplos: ['That last world Sam describes... If those kids get...', 'All these tech guys and their personal insecurity....', "What I really don't understand is , when this syst..."]

Cluster 4 - 274 comentários:
Exemplos: ['3:03 "You can clap about that all you want. Enjoy"...', "I respect everyone's psition on the matter, in my ...", 'The questions should have been more diversified ba...']


### Clusterização Hierárquica (like Mark)

In [ ]:
from IPython.display import HTML, display

# Aplica o CSS (execute apenas uma vez)
HTML("""
<style>
    .output {
        white-space: pre-wrap;
        font-family: monospace;
        line-height: 1.5;
    }
    div.cluster-box {
        border: 1px solid #e0e0e0;
        border-radius: 5px;
        padding: 10px;
        margin: 10px 0;
    }
</style>
""")

# Função de formatação
def format_comment(text, width=120):
    from textwrap import fill
    return fill(text, width=width)


import numpy as np
from sklearn.cluster import MiniBatchKMeans  # Mais eficiente que KMeans tradicional
from ipywidgets import interact, IntSlider
import matplotlib.pyplot as plt
from collections import defaultdict

# 1. Pré-processamento rápido
embeddings_array = np.array([emb for emb in embeddings_array if isinstance(emb, (list, np.ndarray)) and len(emb) > 0])

# 2. Redução de dimensionalidade (opcional, mas ajuda na visualização)
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(embeddings_array)

# 3. Função otimizada para clusterização interativa
def interactive_clustering(n_clusters=5):
    # Clusterização rápida com MiniBatchKMeans
    kmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, batch_size=100)
    labels = kmeans.fit_predict(embeddings_array)
    
    # Visualização simplificada
    plt.figure(figsize=(10, 6))
    scatter = plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], c=labels, cmap='viridis', alpha=0.6, s=10)
    plt.colorbar(scatter)
    plt.title(f"Clusters (k={n_clusters}) - Visualização 2D")
    plt.show()
    
    # Análise textual leve (amostra aleatória)
    unique_labels = np.unique(labels)
    for label in unique_labels:
        cluster_indices = np.where(labels == label)[0]
        sample_idx = np.random.choice(cluster_indices, size=min(6, len(cluster_indices)), replace=False)
        
        print(f"\n🔷 Cluster {int(label)+1} ({len(cluster_indices)} comentários)")
        print("📌 Exemplos:")
        for idx in sample_idx:
            print(f"  - {format_comment(all_comments[idx][:500])}...")

# 4. Interface interativa leve
interact(
    interactive_clustering, 
    n_clusters=IntSlider(min=2, max=20, step=1, value=2, description='Clusters:')
)


interactive(children=(IntSlider(value=2, description='Clusters:', max=20, min=2), Output()), _dom_classes=('wi…

<function __main__.interactive_clustering(n_clusters=5)>